# Trabajo Integrador: un RAG sobre seguridad en IA generativa

**Juan Cifuentes. Tecnicatura Superior en Ciencias de Datos e Inteligencia Artificial, IFTS N.º 24**
* **Materia:** Técnicas de Procesamiento de Habla (Procesamiento de Lenguaje Natural), segundo cuatrimestre de 2026

---

## Cuaderno 04: Probar el RAG antes de entregarlo

## Objetivo

Revisar la recuperación, la respuesta y las fuentes en casos respondibles y sin respuesta.

## Preguntas que ponen a prueba el sistema

Son cinco preguntas: una por cada documento y dos cuyo dato no está en el corpus. Son pocas, así que **no** miden el desempeño real del sistema. El dato esperado lo anoté antes de mirar el resultado, copiado del PDF original.

| Pregunta | Fuente esperada |
|---|---|
| ¿Qué es la inyección de prompt? | OWASP, página 8 |
| ¿Qué exige el artículo 15 del Reglamento de IA sobre ciberseguridad? | AESIA, páginas 16 y 18 |
| ¿Qué medidas se recomiendan para la gestión de identidades y accesos frente a ataques asistidos por IA? | INCIBE, página 5 |
| ¿Cuántos ataques de inyección de prompt hubo en Argentina en 2025? | Ninguna |
| ¿Cuál es el modelo de IA más seguro del mercado? | Ninguna |

In [1]:
import app

app.INDICE = app.preparar_indice()
casos = [
    {"pregunta": "¿Qué es la inyección de prompt?", "evidencia_esperada": "ocurre cuando los prompts del usuario"},
    {"pregunta": "¿Qué exige el artículo 15 del Reglamento de IA sobre ciberseguridad?", "evidencia_esperada": "nivel adecuado de precisión, solidez y ciberseguridad"},
    {"pregunta": "¿Qué medidas se recomiendan para la gestión de identidades y accesos frente a ataques asistidos por IA?", "evidencia_esperada": "autenticación adaptativa y mfa robusto"},
    {"pregunta": "¿Cuántos ataques de inyección de prompt hubo en Argentina en 2025?", "evidencia_esperada": None},
    {"pregunta": "¿Cuál es el modelo de IA más seguro del mercado?", "evidencia_esperada": None},
]

c:\Users\JUANK\Desktop\rag_seguridad_ia\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Primero la recuperación

Compruebo si el dato esperado aparece en alguno de los tres primeros fragmentos. Eso todavía no demuestra que el generador lo vaya a usar bien. Hay que mirar el texto, no sólo el nombre del archivo. La página la veo en la celda siguiente, en «Fuentes enviadas».

In [2]:
for caso in casos:
    hallazgos = app.INDICE.similarity_search_with_score(query=caso["pregunta"], k=3)
    print("\nPregunta:", caso["pregunta"])
    print("Dato esperado:", caso["evidencia_esperada"])
    evidencia_en_contexto = False

    for documento, distancia in hallazgos:
        print("Fuente recuperada:", documento.metadata.get("source"))
        print("Distancia:", round(distancia, 3))
        print("Texto:", documento.page_content[:180])

        if caso["evidencia_esperada"] is not None:
            if caso["evidencia_esperada"] in documento.page_content.lower():
                evidencia_en_contexto = True

    if caso["evidencia_esperada"] is not None:
        print("¿Apareció el dato esperado en el top 3?:", evidencia_en_contexto)


Pregunta: ¿Qué es la inyección de prompt?
Dato esperado: ocurre cuando los prompts del usuario
Fuente recuperada: datos\owasp_top10_llm_2025.pdf
Distancia: 0.598
Texto: OWASP Top 10 para Aplicaciones de LLM 2025
4genai.owasp.org
LLM01:2025 Inyección de prompt
Descripción
Una vulnerabilidad de inyección de prompt ocurre cuando los prompts del usuar
Fuente recuperada: datos\owasp_top10_llm_2025.pdf
Distancia: 0.635
Texto: Inyección de prompt directa
Las inyecciones de prompt directas se producen cuando la entrada de un prompt de
usuario altera directamente el comportamiento del modelo de forma no in
Fuente recuperada: datos\owasp_top10_llm_2025.pdf
Distancia: 0.641
Texto: LLM01:2025 Inyección de prompt  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .   4
Descripción .  .  .  .  .  .  .  .  .  .  .  .
¿Apareció el dato esperado en el top 3?: True

Pregunta: ¿Qué exige el artículo 15 del Reglamento de IA sobre ciberseguridad?
Dato esperado

## Después la generación

Cada caso se clasifica leyendo la respuesta y la evidencia: si respondió con un dato que está en los fragmentos, si citó bien y si se abstuvo en las preguntas sin respuesta. La abstención **no** se deduce de la distancia. Si algo falló, hay que ubicar el paso: extracción, fragmentación, búsqueda o redacción.

In [3]:
for caso in casos:
    respuesta, fuentes = app.consultar(caso["pregunta"])
    print("\nPregunta:", caso["pregunta"])
    print("Respuesta:", respuesta)
    print("Fuentes enviadas:", fuentes[:450])
    print("-" * 60)


Pregunta: ¿Qué es la inyección de prompt?
Respuesta: La inyección de prompt es una vulnerabilidad de seguridad que ocurre cuando los prompts del usuario alteran el comportamiento o la salida del modelo de lenguaje (LLM) de formas no intencionadas, afectando el modelo incluso si las entradas son imperceptibles para los humanos, y estas inyecciones no necesitan ser visibles o leíbles para los humanos, siempre y cuando el contenido sea analizado por el modelo. [1]
Fuentes enviadas: **[1] owasp_top10_llm_2025.pdf, página 8** (distancia 0.598)

```text
OWASP Top 10 para Aplicaciones de LLM 2025
4genai.owasp.org
LLM01:2025 Inyección de prompt
Descripción
Una vulnerabilidad de inyección de prompt ocurre cuando los prompts del usuario alteran el
comportamiento o la salida del LLM en formas no intencionadas. Estas entradas pueden
afectar al modelo incluso si son imperceptibles para los humanos, por lo tanto las inyecciones
de pro
------------------------------------------------------------

Pr

## Lo que encontré al probar

En las tres preguntas respondibles el dato esperado apareció entre los tres primeros fragmentos, y en las dos sin respuesta el modelo se abstuvo. Comparé cada respuesta con la página del PDF.

| Pregunta | Fuente esperada | Fragmento recuperado [1] | Respuesta | Observación |
|---|---|---|---|---|
| ¿Qué es la inyección de prompt? | OWASP, p. 8 | OWASP, p. 8 (0,598) | Da la definición y cita [1]. | Correcta. El fragmento [3] es el índice de OWASP. |
| ¿Qué exige el artículo 15 del Reglamento de IA sobre ciberseguridad? | AESIA, páginas 16 y 18 | AESIA, p. 16 (0,627) | Un nivel adecuado de ciberseguridad durante todo el ciclo de vida. Cita [1]. | Correcta pero incompleta, le falta «precisión y solidez». El fragmento [3] es el índice de AESIA. |
| ¿Qué medidas se recomiendan para la gestión de identidades y accesos frente a ataques asistidos por IA? | INCIBE, p. 5 | INCIBE, p. 5 (0,660) | MFA obligatorio y métodos resistentes al phishing. Cita [1]. | Correcta. Nombra sólo la primera medida de la lista. |
| ¿Cuántos ataques de inyección de prompt hubo en Argentina en 2025? | Ninguna | OWASP, p. 3 (índice) (0,820) | «No encuentro esa información en los documentos.» | Se abstuvo aunque recibió tres fragmentos. |
| ¿Cuál es el modelo de IA más seguro del mercado? | Ninguna | INCIBE, p. 4 (0,907) | «No encuentro esa información en los documentos.» | Se abstuvo. |

Limitaciones que encontré:

1. Los índices de los PDF se recuperan como si fueran contenido. Pasó con OWASP y AESIA, y ocupan lugar en el contexto sin aportar nada. En la app, la misma pregunta del artículo 15 escrita con comillas y un punto trajo el índice de AESIA en primer lugar, y el modelo lo citó como [1]. Lo cambiaría quitando esas páginas antes de indexar.
2. La abstención depende del prompt. Chroma siempre devuelve fragmentos, aunque la pregunta no tenga respuesta en el corpus. En las pruebas el modelo se abstuvo, pero no está garantizado.

## Glosario

| Término | Qué significa acá |
|---|---|
| **Caso respondible** | Pregunta cuya respuesta está respaldada por uno de los documentos publicados. |
| **Caso sin respuesta** | Pregunta cuyo dato no figura en el corpus y obliga a reconocer un límite. |
| **Falla de recuperación** | El fragmento pertinente no llega al contexto del generador. |
| **Falla de generación** | La evidencia llegó, pero el modelo la interpretó o la citó mal. |